# Simulation Complète du Modèle de Zooplancton avec Transport

Ce notebook démontre le système complet SEAPOPYM-MESSAGE avec :

-   **Biologie** : Modèle de zooplancton 2 compartiments (biomasse adulte + production juvénile structurée en âge)
-   **Transport** : Advection + diffusion pour biomasse ET production
-   **Forcings** : NPP, température + **derived forcings** (tau_r, mortality calculés depuis T)
-   **Architecture** : API générique avec TransportConfig + ForcingManager

## Modèle biologique

1. **Production juvénile** p(τ) [11 classes d'âge] :

    - Source : p(0) = E × NPP
    - Vieillissement avec absorption au recrutement
    - Absorption totale (α→∞) : âge ≥ τ_r(T)

2. **Biomasse adulte** B :

    - dB/dt = R - λ(T)B
    - R = recrutement depuis production
    - λ(T) = mortalité dépendante de la température (derived forcing)

3. **Transport** :
    - Advection + diffusion pour B et p
    - Grille sphérique
    - Conditions limites périodiques (longitude), fermées (latitude)

## Derived Forcings (nouveauté)

-   **tau_r(T)** : Âge minimum de recrutement dépendant de la température
-   **mortality(T)** : Taux de mortalité dépendant de la température

Calculés **1× par timestep** par le ForcingManager (optimisation) puis distribués à tous les workers.

## Ordre d'exécution critique

⚠️ Les units biologiques doivent s'exécuter dans cet ordre :

1. `compute_recruitment` → 2. `age_production` → 3. `update_biomass`

Sinon : recruitment = 0 (car production déjà absorbée) !


In [ ]:
# Imports
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
import ray
import xarray as xr

from seapopym_message.core.kernel import Kernel
from seapopym_message.distributed.scheduler import EventScheduler
from seapopym_message.distributed.transport_config import FieldConfig, TransportConfig
from seapopym_message.distributed.worker import CellWorker2D
from seapopym_message.forcing.manager import ForcingManager
from seapopym_message.kernels.zooplankton import (
    age_production,
    compute_mortality_forcing,
    compute_recruitment,
    compute_tau_r_forcing,
    update_biomass,
)
from seapopym_message.transport.worker import TransportWorker
from seapopym_message.utils.grid import GridInfo

print("✅ Modules importés avec succès !")

## 1. Configuration du système distribué


In [ ]:
# Initialiser Ray
if not ray.is_initialized():
    ray.init(num_cpus=4, ignore_reinit_error=True)
    print("✅ Ray initialisé")
else:
    print("✅ Ray déjà initialisé")

In [ ]:
# Paramètres de la grille globale (région équatoriale Pacifique)
lat_min, lat_max = -20.0, 20.0  # 40° de latitude
lon_min, lon_max = -20.0, 20.0  # 40° de longitude (Pacifique central)
global_nlat, global_nlon = 20, 20  # Résolution ~2°

# Configuration des workers (2x2 = 4 workers)
n_workers_lat, n_workers_lon = 2, 2
nlat_per_worker = global_nlat // n_workers_lat
nlon_per_worker = global_nlon // n_workers_lon

# Paramètres biologiques
n_ages = 11  # Nombre de classes d'âge

print(f"Grille globale: {global_nlat}×{global_nlon}")
print(f"Domaine spatial: [{lat_min}, {lat_max}]° × [{lon_min}, {lon_max}]°")
print(f"Workers: {n_workers_lat}×{n_workers_lon} = {n_workers_lat * n_workers_lon} total")
print(f"Taille par worker: {nlat_per_worker}×{nlon_per_worker}")
print(f"Classes d'âge: {n_ages}")

## 2. Création des forcings (NPP et Température)


In [ ]:
# Grille de latitude/longitude
lats = np.linspace(lat_min, lat_max, global_nlat)
lons = np.linspace(lon_min, lon_max, global_nlon)
LON, LAT = np.meshgrid(lons, lats)

# 1. TEMPÉRATURE : constante à 0°C partout
temperature = np.zeros_like(LON)

# 2. NPP : barre horizontale au centre (maximum à lat=0)
NPP_base = 300 / 1000 / 1000  # kg/m²/day
sigma_lat = 5.0  # Largeur de la barre [degrés]
npp = NPP_base * np.exp(-(LAT**2) / (2 * sigma_lat**2))

# Visualisation
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Température
im1 = axes[0].contourf(LON, LAT, temperature, levels=15, cmap="RdYlBu_r")
axes[0].set_title("Température de surface [°C]", fontsize=14, fontweight="bold")
axes[0].set_xlabel("Longitude [°]")
axes[0].set_ylabel("Latitude [°]")
plt.colorbar(im1, ax=axes[0], label="°C")

# NPP
im2 = axes[1].contourf(LON, LAT, npp, levels=15, cmap="YlGn")
axes[1].set_title("Production Primaire Nette (NPP) [kg/m²/day]", fontsize=14, fontweight="bold")
axes[1].set_xlabel("Longitude [°]")
axes[1].set_ylabel("Latitude [°]")
plt.colorbar(im2, ax=axes[1], label="kg/m²/day")

plt.tight_layout()
plt.show()

print(f"Température : {temperature.min():.1f}°C (constante)")
print(f"NPP : [{npp.min():.2e}, {npp.max():.2e}] kg/m²/day (barre horizontale centrée)")

## 3. Paramètres du modèle de zooplancton


In [ ]:
# Paramètres du modèle biologique (pour les units)
# Note : tau_r et mortality sont maintenant calculés par derived forcings
zoo_params = {
    # Classes d'âge
    "n_ages": n_ages,
    # Efficacité de transfert NPP → Production
    "E": 0.1668,  # ~16.7% du NPP devient production de zooplancton
}

print("Paramètres biologiques (units) :")
for key, value in zoo_params.items():
    print(f"  {key}: {value}")

print("\nNote : Les paramètres de température (tau_r0, gamma_tau_r, lambda_0, gamma_lambda, T_ref)")
print("       seront passés au ForcingManager pour calculer les derived forcings.")

## 4. Création du kernel biologique


In [ ]:
# Créer le kernel avec les 3 unités du modèle zooplancton
# ⚠️ ORDRE CRITIQUE : compute_recruitment DOIT être avant age_production
# Raison : compute_recruitment lit production[age-1] AVANT que age_production
#          ne les absorbe (set à 0). Si inversé, recruitment = 0 !
kernel = Kernel(
    [
        compute_recruitment,  # 1. DOIT être premier : calcule R depuis production
        age_production,  # 2. DOIT être second : vieillit et absorbe
        update_biomass,  # 3. DOIT être troisième : applique R à biomasse
    ]
)

print("✅ Kernel biologique créé avec 3 unités (ordre critique) :")
print("  1. compute_recruitment : R = Σ production[age-1] pour age ≥ τ_r")
print("  2. age_production : vieillissement + absorption + NPP source")
print("  3. update_biomass : B^{n+1} = (B + R×dt) / (1 + λ×dt)")

## 5. Initialisation des champs biologiques


In [ ]:
# État initial : biomasse et production à zéro (production sera générée par NPP)
biomass_init_jax = jnp.zeros((global_nlat, global_nlon), dtype=jnp.float32)
production_init_jax = jnp.zeros((n_ages, global_nlat, global_nlon), dtype=jnp.float32)

# Visualisation
fig, ax = plt.subplots(1, 1, figsize=(12, 6))
im = ax.contourf(LON, LAT, biomass_init_jax, levels=15, cmap="YlOrRd")
ax.set_title("Biomasse Adulte Initiale [kg/m²]", fontsize=14, fontweight="bold")
ax.set_xlabel("Longitude [°]")
ax.set_ylabel("Latitude [°]")
plt.colorbar(im, ax=ax, label="kg/m²")
plt.tight_layout()
plt.show()

mass_init_biomass = float(jnp.sum(biomass_init_jax))
mass_init_production = float(jnp.sum(production_init_jax))
print(f"Masse initiale biomasse : {mass_init_biomass:.2e} kg/m²")
print(f"Masse initiale production : {mass_init_production:.2e} kg/m²")
print("Production sera générée par la source NPP durant la simulation")

## 6. Création des CellWorkers


In [ ]:
# GridInfo pour les workers
grid_info = GridInfo(
    lat_min=lat_min,
    lat_max=lat_max,
    lon_min=lon_min,
    lon_max=lon_max,
    nlat=global_nlat,
    nlon=global_nlon,
)

# Créer les workers
workers = []
for i in range(n_workers_lat):
    for j in range(n_workers_lon):
        lat_start = i * nlat_per_worker
        lat_end = (i + 1) * nlat_per_worker
        lon_start = j * nlon_per_worker
        lon_end = (j + 1) * nlon_per_worker

        worker = CellWorker2D.remote(
            worker_id=i * n_workers_lon + j,
            grid_info=grid_info,
            lat_start=lat_start,
            lat_end=lat_end,
            lon_start=lon_start,
            lon_end=lon_end,
            kernel=kernel,
            params=zoo_params,
        )

        # Initialiser l'état avec les portions correspondantes
        biomass_patch = biomass_init_jax[lat_start:lat_end, lon_start:lon_end]
        production_patch = production_init_jax[:, lat_start:lat_end, lon_start:lon_end]

        ray.get(
            worker.set_initial_state.remote(
                {"biomass": biomass_patch, "production": production_patch}
            )
        )

        workers.append(worker)

print(f"✅ {len(workers)} CellWorkers créés et initialisés")

## 7. Création du TransportWorker et configuration


In [ ]:
# Créer le TransportWorker (grille sphérique)
transport_worker = TransportWorker.remote(
    grid_type="spherical",
    lat_min=lat_min,
    lat_max=lat_max,
    lon_min=lon_min,
    lon_max=lon_max,
    nlat=global_nlat,
    nlon=global_nlon,
    lat_bc="closed",  # Fermé aux pôles
    lon_bc="closed",  # Périodique en longitude
)

# Configuration du transport : 2 champs (biomass 2D + production 3D)
transport_config = TransportConfig(
    fields=[
        FieldConfig(name="biomass", dims=["Y", "X"]),  # 2D
        FieldConfig(name="production", dims=["age", "Y", "X"]),  # 3D
    ]
)

print("✅ TransportWorker créé (grille sphérique)")
print("✅ TransportConfig créé pour 2 champs :")
print("  - biomass (2D) : Y × X")
print("  - production (3D) : age × Y × X")

## 8. Préparation des forcings et paramètres de transport


In [ ]:
# Champs de vitesse : COURANT CIRCULAIRE autour du centre

# Centre de la circulation (centre du domaine)
lat_center = 0.0
lon_center = 0.0

# Distance au centre pour chaque point
dx = LON - lon_center
dy = LAT - lat_center
r = np.sqrt(dx**2 + dy**2)  # Distance radiale [degrés]

# Profil de vitesse : maximum à r_peak, décroissance gaussienne
r_peak = 10.0  # Rayon du maximum de vitesse [degrés]
sigma_r = 8.0  # Largeur du gyre [degrés]
v_max = 1.0  # Vitesse tangentielle maximale [m/s]

# Vitesse tangentielle (profil gaussien centré sur r_peak)
v_tangent = v_max * np.exp(-((r - r_peak) ** 2) / (2 * sigma_r**2))

# Éviter division par zéro au centre
r_safe = np.where(r < 1e-6, 1e-6, r)

# Composantes cartésiennes (rotation ANTIHORAIRE = cyclonique dans hémisphère Nord)
# u = composante zonale (est-ouest)
# v = composante méridienne (nord-sud)
u_global = (-v_tangent * dy / r_safe).astype(np.float32)
v_global = (v_tangent * dx / r_safe).astype(np.float32)

u_global_jax = jnp.array(u_global, dtype=jnp.float32)
v_global_jax = jnp.array(v_global, dtype=jnp.float32)

# Coefficient de diffusion
D = 1000.0  # m²/s (diffusion méso-échelle)

# Pas de temps (1 jour)
dt = 24 * 60 * 60  # secondes

# Vérifier CFL
# Pour grille sphérique, dx varie avec latitude
# Approximation : dx ~ 111 km/° × Δlon
dlon = (lon_max - lon_min) / global_nlon
dlat = (lat_max - lat_min) / global_nlat
dx_approx = 111e3 * dlon  # mètres (à l'équateur)
dy_approx = 111e3 * dlat  # mètres

u_max = np.abs(u_global).max()
v_max_array = np.abs(v_global).max()
vel_max = np.sqrt(u_max**2 + v_max_array**2)

cfl_adv = vel_max * dt / min(dx_approx, dy_approx)
cfl_diff = D * dt / min(dx_approx, dy_approx) ** 2

# Visualiser le champ de vitesse
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# Magnitude de la vitesse
velocity_magnitude = np.sqrt(u_global**2 + v_global**2)
im1 = axes[0].contourf(LON, LAT, velocity_magnitude, levels=15, cmap="YlOrRd")
axes[0].set_title("Magnitude de la vitesse [m/s]", fontsize=14, fontweight="bold")
axes[0].set_xlabel("Longitude [°]")
axes[0].set_ylabel("Latitude [°]")
plt.colorbar(im1, ax=axes[0], label="m/s")

# Champ de vecteurs (sous-échantillonné pour lisibilité)
skip = 2  # Afficher 1 vecteur sur 2
axes[1].quiver(
    LON[::skip, ::skip],
    LAT[::skip, ::skip],
    u_global[::skip, ::skip],
    v_global[::skip, ::skip],
    velocity_magnitude[::skip, ::skip],
    cmap="YlOrRd",
    scale=5,
    width=0.004,
)
axes[1].set_title("Champ de vitesse (vecteurs)", fontsize=14, fontweight="bold")
axes[1].set_xlabel("Longitude [°]")
axes[1].set_ylabel("Latitude [°]")
axes[1].set_aspect("equal")

plt.tight_layout()
plt.show()

print(f"Courant circulaire autour de ({lat_center}°, {lon_center}°)")
print(f"  Rayon du maximum : r_peak = {r_peak}°")
print(f"  Largeur du gyre : σ = {sigma_r}°")
print(f"  Vitesse maximale : v_max = {v_max} m/s")
print(f"\nVitesse : u ∈ [{u_global.min():.2f}, {u_global.max():.2f}] m/s")
print(f"          v ∈ [{v_global.min():.2f}, {v_global.max():.2f}] m/s")
print(f"          |v| ∈ [0, {velocity_magnitude.max():.2f}] m/s")
print(f"Diffusion : D={D:.1f} m²/s")
print(f"Pas de temps : dt={dt:.0f} s = {dt / 3600:.1f} h")
print(f"Résolution spatiale : dx≈{dx_approx / 1e3:.1f} km, dy≈{dy_approx / 1e3:.1f} km")
print(f"\nCFL advection : {cfl_adv:.4f} {'✅' if cfl_adv < 1 else '⚠️'}")
print(f"CFL diffusion : {cfl_diff:.4f} {'✅' if cfl_diff < 0.25 else '⚠️'}")

In [ ]:
# Créer des datasets xarray pour les forcings constants
# Le ForcingManager attend des xarray.Dataset avec dimension temps

# Durée de simulation
days = 30 * 12
t_total = days * 24 * 60 * 60  # secondes
times = np.arange(0, days + 1) * 24 * 3600.0  # En secondes

# Créer datasets xarray (forcings constants répétés sur tous les temps)
temp_ds = xr.Dataset(
    {
        "temperature": (
            ["time", "lat", "lon"],
            np.repeat(temperature[np.newaxis, :, :], len(times), axis=0),
        )
    },
    coords={"time": times, "lat": lats, "lon": lons},
    attrs={"units": "°C", "interpolation_method": "linear"},
)

npp_ds = xr.Dataset(
    {"npp": (["time", "lat", "lon"], np.repeat(npp[np.newaxis, :, :], len(times), axis=0))},
    coords={"time": times, "lat": lats, "lon": lons},
    attrs={"units": "kg/m²/day", "interpolation_method": "linear"},
)

u_ds = xr.Dataset(
    {"u": (["time", "lat", "lon"], np.repeat(u_global[np.newaxis, :, :], len(times), axis=0))},
    coords={"time": times, "lat": lats, "lon": lons},
    attrs={"units": "m/s", "interpolation_method": "linear"},
)

v_ds = xr.Dataset(
    {"v": (["time", "lat", "lon"], np.repeat(v_global[np.newaxis, :, :], len(times), axis=0))},
    coords={"time": times, "lat": lats, "lon": lons},
    attrs={"units": "m/s", "interpolation_method": "linear"},
)

# Créer le ForcingManager avec les datasets
forcing_manager = ForcingManager(
    datasets={
        "temperature": temp_ds,
        "npp": npp_ds,
        "u": u_ds,
        "v": v_ds,
    }
)

# Enregistrer les derived forcings
forcing_manager.register_derived(compute_tau_r_forcing)
forcing_manager.register_derived(compute_mortality_forcing)

# Paramètres pour les derived forcings
forcing_params = {
    "tau_r0": 10,
    "gamma_tau_r": 0.05,
    "T_ref": 0.0,
    "lambda_0": 0,
    "gamma_lambda": 0.05,
}

print("✅ ForcingManager créé avec :")
print("  - 4 forcings de base : temperature, npp, u, v")
print("  - 2 derived forcings : tau_r, mortality")
print(f"  - Plage temporelle : 0 à {times[-1] / (24 * 3600):.0f} jours")

## 9. Création de l'EventScheduler et lancement de la simulation


In [ ]:
# Wrapper pour passer forcing_params à chaque appel
class ForcingManagerWrapper:
    """Wrapper qui passe forcing_params au ForcingManager."""

    def __init__(self, forcing_manager, forcing_params):
        self.forcing_manager = forcing_manager
        self.forcing_params = forcing_params

    def prepare_timestep_distributed(self, time, params):
        """Délègue au ForcingManager avec forcing_params."""
        # Merger les params (scheduler params + forcing params)
        all_params = {**self.forcing_params, **params}
        return self.forcing_manager.prepare_timestep_distributed(time, all_params)


# Créer le wrapper
forcing_manager_wrapper = ForcingManagerWrapper(forcing_manager, forcing_params)

# Créer le scheduler
scheduler = EventScheduler(
    workers=workers,
    dt=dt,
    t_max=t_total,
    forcing_manager=forcing_manager_wrapper,
    transport_worker=transport_worker,
    transport_config=transport_config,
    global_nlat=global_nlat,
    global_nlon=global_nlon,
    forcing_params={"horizontal_diffusivity": D, "age": n_ages},
)

print("✅ EventScheduler créé avec :")
print("  - ForcingManager avec derived forcings (tau_r, mortality)")
print("  - Transport activé (advection + diffusion)")
print(f"  - Durée simulation : {t_total / (24 * 3600):.0f} jours")
print(f"  - Nombre de timesteps : {int(t_total / dt)}")

In [ ]:
# Configuration des snapshots
N_snapshots = 7  # Snapshots à t=0, 5, 10, 15, 20, 25, 30 jours
snapshot_days = np.linspace(0, days, N_snapshots)
snapshot_times = snapshot_days * 24 * 60 * 60  # En secondes

print(f"Snapshots sauvegardés à : {snapshot_days} jours")

# Stockage des snapshots
biomass_snapshots = []
production_total_snapshots = []  # Somme sur âges
time_snapshots = []
conservation_history = {"biomass": [], "production": []}
mass_history = {"biomass": [], "production": [], "total": []}

# Sauvegarder l'état initial (t=0)
biomass_snapshots.append(np.array(biomass_init_jax))
production_total_snapshots.append(np.zeros((global_nlat, global_nlon)))
time_snapshots.append(0.0)
mass_history["biomass"].append(mass_init_biomass)
mass_history["production"].append(mass_init_production)
mass_history["total"].append(mass_init_biomass + mass_init_production)
conservation_history["biomass"].append(100.0)
conservation_history["production"].append(100.0)

print("\n🚀 Début de la simulation...")

next_snapshot_idx = 1

while scheduler.t_current < t_total - 1e-6:
    # Exécuter un step
    diagnostics = scheduler.step()

    # Vérifier si on doit sauvegarder un snapshot
    if (
        next_snapshot_idx < N_snapshots
        and scheduler.t_current >= snapshot_times[next_snapshot_idx] - dt / 2
    ):
        # Collecter les champs globaux
        biomass_global = scheduler._collect_global_field("biomass", (global_nlat, global_nlon))
        production_global = scheduler._collect_global_field(
            "production", (n_ages, global_nlat, global_nlon)
        )

        # Calculer production totale (somme sur âges)
        production_total = jnp.sum(production_global, axis=0)

        # Sauvegarder
        biomass_snapshots.append(np.array(biomass_global))
        production_total_snapshots.append(np.array(production_total))
        time_snapshots.append(scheduler.t_current / (24 * 3600))  # jours

        # Masses
        mass_biomass = float(jnp.sum(biomass_global))
        mass_production = float(jnp.sum(production_global))
        mass_total = mass_biomass + mass_production

        mass_history["biomass"].append(mass_biomass)
        mass_history["production"].append(mass_production)
        mass_history["total"].append(mass_total)

        # Conservation (par rapport à l'état initial)
        cons_biomass = (mass_biomass / mass_init_biomass) * 100 if mass_init_biomass > 0 else 100
        cons_production = (
            (mass_production / mass_init_production) * 100 if mass_init_production > 0 else 100
        )

        conservation_history["biomass"].append(cons_biomass)
        conservation_history["production"].append(cons_production)

        print(
            f"  Snapshot {next_snapshot_idx}/{N_snapshots - 1} - "
            f"t={time_snapshots[-1]:.1f} jours - "
            f"Masse totale: {mass_total:.2e}"
        )

        next_snapshot_idx += 1

print("\n✅ Simulation terminée !")

## 10. Visualisation de l'évolution de la biomasse


In [ ]:
# Créer une figure avec tous les snapshots de biomasse
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()

# Échelle de couleur cohérente
vmax_biomass = max(b.max() for b in biomass_snapshots)

for i, (biomass_snap, time_snap) in enumerate(zip(biomass_snapshots, time_snapshots)):
    if i >= len(axes):
        break

    im = axes[i].contourf(
        LON, LAT, biomass_snap, levels=15, cmap="YlOrRd", vmin=0, vmax=vmax_biomass
    )
    axes[i].set_title(f"Biomasse - t = {time_snap:.1f} jours", fontsize=12, fontweight="bold")
    axes[i].set_xlabel("Longitude [°]")
    axes[i].set_ylabel("Latitude [°]")
    plt.colorbar(im, ax=axes[i], label="kg/m²", fraction=0.046)

# Masquer le dernier subplot si impair
if len(biomass_snapshots) < len(axes):
    axes[-1].axis("off")

plt.suptitle(
    "Évolution de la Biomasse Adulte (B) - Modèle Zooplancton + Transport",
    fontsize=16,
    fontweight="bold",
)
plt.tight_layout()
plt.show()

## 11. Visualisation de l'évolution de la production totale


In [ ]:
# Créer une figure avec tous les snapshots de production
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()

# Échelle de couleur cohérente
vmax_production = max(p.max() for p in production_total_snapshots)

for i, (production_snap, time_snap) in enumerate(zip(production_total_snapshots, time_snapshots)):
    if i >= len(axes):
        break

    im = axes[i].contourf(
        LON, LAT, production_snap, levels=15, cmap="Greens", vmin=0, vmax=vmax_production
    )
    axes[i].set_title(
        f"Production Totale - t = {time_snap:.1f} jours", fontsize=12, fontweight="bold"
    )
    axes[i].set_xlabel("Longitude [°]")
    axes[i].set_ylabel("Latitude [°]")
    plt.colorbar(im, ax=axes[i], label="kg/m²", fraction=0.046)

# Masquer le dernier subplot si impair
if len(production_total_snapshots) < len(axes):
    axes[-1].axis("off")

plt.suptitle(
    "Évolution de la Production Juvénile Totale (Σp) - Modèle Zooplancton + Transport",
    fontsize=16,
    fontweight="bold",
)
plt.tight_layout()
plt.show()

## 12. Analyse temporelle des masses


In [ ]:
# Évolution des masses
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Masse totale (biomasse + production)
axes[0].plot(time_snapshots, mass_history["total"], "k-o", linewidth=2, markersize=8, label="Total")
axes[0].plot(
    time_snapshots, mass_history["biomass"], "r-o", linewidth=2, markersize=6, label="Biomasse"
)
axes[0].plot(
    time_snapshots,
    mass_history["production"],
    "g-o",
    linewidth=2,
    markersize=6,
    label="Production",
)
axes[0].set_xlabel("Temps [jours]", fontsize=12)
axes[0].set_ylabel("Masse [kg/m² × nb cellules]", fontsize=12)
axes[0].set_title("Évolution des Masses", fontsize=14, fontweight="bold")
axes[0].grid(True, alpha=0.3)
axes[0].legend(fontsize=10)

# Rapport Production / Biomasse
ratio = np.array(mass_history["production"]) / (np.array(mass_history["biomass"]) + 1e-10)
axes[1].plot(time_snapshots, ratio, "b-o", linewidth=2, markersize=8)
axes[1].set_xlabel("Temps [jours]", fontsize=12)
axes[1].set_ylabel("Ratio Production / Biomasse", fontsize=12)
axes[1].set_title("Ratio Production/Biomasse", fontsize=14, fontweight="bold")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n📊 Statistiques finales (jour {time_snapshots[-1]:.1f}) :")
print(f"  - Masse totale initiale : {mass_history['total'][0]:.2e}")
print(f"  - Masse totale finale : {mass_history['total'][-1]:.2e}")

## 13. Visualisation comparée : État initial vs État final


In [ ]:
# Comparaison état initial / final
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Biomasse initiale
im1 = axes[0, 0].contourf(LON, LAT, biomass_snapshots[0], levels=15, cmap="YlOrRd")
axes[0, 0].set_title(f"Biomasse - t = 0 jours", fontsize=14, fontweight="bold")
axes[0, 0].set_xlabel("Longitude [°]")
axes[0, 0].set_ylabel("Latitude [°]")
plt.colorbar(im1, ax=axes[0, 0], label="kg/m²")

# Biomasse finale
im2 = axes[0, 1].contourf(LON, LAT, biomass_snapshots[-1], levels=15, cmap="YlOrRd")
axes[0, 1].set_title(
    f"Biomasse - t = {time_snapshots[-1]:.1f} jours", fontsize=14, fontweight="bold"
)
axes[0, 1].set_xlabel("Longitude [°]")
axes[0, 1].set_ylabel("Latitude [°]")
plt.colorbar(im2, ax=axes[0, 1], label="kg/m²")

# Production initiale
im3 = axes[1, 0].contourf(LON, LAT, production_total_snapshots[0], levels=15, cmap="Greens")
axes[1, 0].set_title(f"Production Totale - t = 0 jours", fontsize=14, fontweight="bold")
axes[1, 0].set_xlabel("Longitude [°]")
axes[1, 0].set_ylabel("Latitude [°]")
plt.colorbar(im3, ax=axes[1, 0], label="kg/m²")

# Production finale
im4 = axes[1, 1].contourf(LON, LAT, production_total_snapshots[-1], levels=15, cmap="Greens")
axes[1, 1].set_title(
    f"Production Totale - t = {time_snapshots[-1]:.1f} jours", fontsize=14, fontweight="bold"
)
axes[1, 1].set_xlabel("Longitude [°]")
axes[1, 1].set_ylabel("Latitude [°]")
plt.colorbar(im4, ax=axes[1, 1], label="kg/m²")

plt.suptitle(
    "Comparaison État Initial vs État Final - Modèle Zooplancton Complet",
    fontsize=16,
    fontweight="bold",
)
plt.tight_layout()
plt.show()

## 14. Résumé et conclusions

Cette simulation a démontré le système complet **SEAPOPYM-MESSAGE** avec :

### Architecture

-   **EventScheduler** : Coordination des 5 phases (Biologie → Collecte → Transport → Redistribution → Agrégation)
-   **CellWorker2D** (4 workers) : Calculs biologiques distribués en parallèle
-   **TransportWorker** : Transport centralisé sur grille sphérique
-   **ForcingManager** : Gestion des forcings avec derived forcings (tau_r, mortality)
-   **API générique** : TransportConfig avec FieldConfig pour support multi-champs

### Modèle biologique (3 unités - ordre critique)

1.  **compute_recruitment** : Calcule R = Σ production[age-1] pour age ≥ τ_r (DOIT être premier)
2.  **age_production** : Vieillissement + absorption au recrutement + source NPP (DOIT être second)
3.  **update_biomass** : B^{n+1} = (B + R×dt) / (1 + λ×dt) (DOIT être troisième)

⚠️ **Ordre critique** : `compute_recruitment` lit `production[age-1]` AVANT que `age_production` ne l'absorbe (set à 0)

### Derived Forcings

-   **tau_r**(T) = τ*r0 × exp(-γ*τr × (T - T_ref)) : Âge minimum de recrutement
-   **mortality**(T) = λ₀ × exp(γ_λ × (T - T_ref)) : Taux de mortalité

Calculés **1× par timestep** par le ForcingManager, puis distribués à tous les workers (optimisation)

### Transport physique

-   Advection : Courant équatorial vers l'ouest + circulation méridienne
-   Diffusion : D=1000 m²/s (méso-échelle)
-   Support multi-dimensionnel : biomass (2D) + production (3D avec âge)

### Résultats attendus

-   ✅ Génération de production depuis NPP équatorial
-   ✅ Recrutement vers biomasse adulte avec âge dépendant de T
-   ✅ Mortalité dépendante de la température
-   ✅ Transport advectif + diffusif des 2 compartiments
-   ✅ Distribution spatiale influencée par température via derived forcings
-   ✅ Équilibre dynamique entre production, recrutement, mortalité et transport


In [ ]:
# Arrêter Ray (optionnel)
ray.shutdown()
# print("Ray arrêté")